In [ ]:
from datascience import *
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')
import numpy as np
import warnings
warnings.simplefilter(action='ignore')

## Lecture 12 ##

## A Join Example ##

In [ ]:
full = Table.read_table('nc-est2019-agesex-res.csv')
census = full.select('SEX', 'AGE', 'POPESTIMATE2019')
census.show(3)

In [ ]:
sex_codes = Table().with_columns(
    'SEX CODE', make_array(0, 1, 2),
    'CODE DEFINITION', make_array('All', 'Selected Male', 'Selected Female')
)
sex_codes

In [ ]:
sex_codes.join('SEX CODE', census, 'SEX').sort('AGE').show(3)

## E-Scooter Trips in Minneapolis, July 2019 ##

In [ ]:
trips = Table.read_table('EScooterTrips_Jul2019.csv')
trips

## Distribution of Durations ##

In [ ]:
trips.hist('TripDuration')
plt.show();

In [ ]:
trips.sort('TripDuration', descending=True).show(3)

In [ ]:
25188 / 60 / 60 # seconds to hours

In [ ]:
plausible_trips = trips.where('TripDuration', are.below(3600)) # below an hour

In [ ]:
plausible_trips.hist('TripDuration')
plt.show(); # negative trips???

In [ ]:
plausible_trips = plausible_trips.where('TripDuration', are.above_or_equal_to(0))
plausible_trips.hist('TripDuration')
plt.show();

In [ ]:
plausible_trips.hist('TripDuration', bins=np.arange(0, 3600, 200), unit='Second')
plt.show();

In [ ]:
# Approx percent of people who have 
# a ride duration between 200 and 400 seconds
# "between" = [200, 400) 

(400 - 200) * 0.13

In [ ]:
plausible_trips.where('TripDuration', are.between(200, 400)).num_rows

In [ ]:
plausible_trips.num_rows

In [ ]:
# pretty close!!
34786 / 134939

In [ ]:
plausible_trips.hist('TripDuration', bins=60, unit='Second')
plt.show();

In [ ]:
# is TripDistance associated with TripDuration?
plausible_trips.scatter('TripDistance', 'TripDuration')
plt.show();

In [ ]:
# almost 40,000 meters?? that's 40 km! a 10k is like 6.2 miles, so 40km is like 25 miles for reference
plausible_trips.sort('TripDistance', descending=True).show(3)

## Start and End Points ##

In [ ]:
# Most common start point
starts = plausible_trips.group('StartAddress').sort('count', descending=True)
starts

In [ ]:
# Note: some of these are 'hypothetical' addresses - the city keeps track of addresses even when there are no structures there
# Normally, you can input these addresses into Google Maps and it will show you where they are

In [ ]:
# Let's focus on trips either starting or ending at the 10 most common start points
most_common_starts = starts.take(np.arange(0, 10)).column('StartAddress')
plausible_trips_with_common_starts = plausible_trips.where('StartAddress', are.contained_in(most_common_starts)).where('EndAddress', are.contained_in(most_common_starts))
plausible_trips_with_common_starts

In [ ]:
# Number of trips between locations
plausible_trips_with_common_starts.pivot('StartAddress', 'EndAddress')

In [ ]:
# Average durations of trips between stations
plausible_trips_with_common_starts.pivot('StartAddress', 'EndAddress', values='TripDuration', collect=np.average)

## Fastest Trips between Locations ##

How can we find the fastest trip ever between each pair of addresses?

In [ ]:
# Let's pare down our dataset a bit - I don't want to crash the server running this notebook
n = 100
top_n_common_starts = starts.take(np.arange(n)).column('StartAddress')
common_trips = plausible_trips.where('StartAddress', are.contained_in(top_n_common_starts)).where('EndAddress', are.contained_in(top_n_common_starts))
common_trips

In [ ]:
duration = common_trips.select('StartAddress', 'EndAddress', 'TripDuration')
duration

In [ ]:
shortest = duration.group(['StartAddress', 'EndAddress'], min)
shortest.show(5)

## Practice question

Find the 5 locations closest to 0 WASHINGTON AVE S by minimum trip time.

In [ ]:
from_washington_ave_s = shortest.where(
    'StartAddress', are.equal_to('0 WASHINGTON AVE S')).sort(
    'TripDuration min')
from_washington_ave_s.take(np.arange(5))

## Maps ##

In [ ]:
geo_data = Table.read_table('mpls_scooter_addresses_with_lat_long.csv')
geo_data

In [ ]:
# you don't need to know the details of this cell - the mapping module needs the data in a very particular format
map_data = geo_data.where('Latitude', are.between(40, 48)).where('Longitude', are.between(-99, -70)) # clean out the nans
map_data = map_data.relabeled('Latitude', 'lat').relabeled('Longitude', 'long').relabeled('Address', 'labels') # rename columns to what the mapping module expects
map_data = map_data.select('lat', 'long', 'labels') # only keep the columns the mapping module needs

In [ ]:
Marker.map_table(map_data.take(np.arange(100)))

### Practice question

Map all locations within 4 minutes (minimum ride time) of 0 WASHINGTON AVE S.

In [ ]:
from_washington_ave_s

In [ ]:
close_from_wash = from_washington_ave_s.where('TripDuration min', are.below(4 * 60)) 
# don't be fooled by 'min' - it means minimum, not minutes :)
close_from_wash

In [ ]:
map_data

In [ ]:
joined = map_data.join('labels', close_from_wash, 'EndAddress')
joined

In [ ]:
close_map_data = joined.select('lat', 'long', 'labels') # this is the format the mapping module expects
close_map_data

In [ ]:
Marker.map_table(close_map_data)

Choose marker colors by the minimum time from Washington Ave

In [ ]:
minutes = np.round(from_washington_ave_s.column("TripDuration min") / 60)
print(min(minutes), max(minutes))

In [ ]:
colors = Table().with_columns(
    "minutes", np.arange(max(minutes)),
    "colors",  ["darkblue", "blue", "lightblue", 
                "lightgreen", "green", "darkgreen",
                "yellow", "orange", "red",
                "darkred", "gray", "gray", 
                "gray", "gray", "gray",
                "gray", "gray", "gray", "gray"])
colors_wash = (from_washington_ave_s
 .with_column("Minutes", minutes)
 .join("Minutes", colors, "minutes"))

colored_markers = (map_data
      .join('labels', colors_wash, 'EndAddress')
      .select('lat', 'long', 'labels', 'colors'))
Marker.map_table(colored_markers)

In [ ]:
# Plot all locations with size corresponding to number of trips starting there
map_starts = map_data.join('labels', starts, 'StartAddress').sort('count', descending=True)
map_starts.show(3)

In [ ]:
data_to_plot = map_starts.select('lat', 'long', 'labels').with_columns(
    'colors', 'blue',
    'areas', map_starts.column('count')
)
data_to_plot.show(3)

In [ ]:
# Before you run this cell, clear output for other map cells to avoid crashing the notebook
Circle.map_table(data_to_plot.take(np.arange(300))) # limit to 300 points for performance